# Step 2: metadata and quality checks

Step 1 downloaded FASTQ files. This notebook does two things with them.

**A. Keep the metadata next to the data.** A file that nobody can explain is useless. For every FASTQ file this notebook fetches ENA's records (run, experiment, sample, study), saves them in the `metadata/` folder, and builds one table, `metadata.tsv`, with one row per file.

**B. Check the quality of the reads and clean them.** It runs FastQC, then a second tool called fastplong, then trims and filters the reads.

What it does, in order:

1. Lists the FASTQ files in the `data/<accession>/` folder.
2. Writes a `README.txt` there that explains every folder.
3. Fetches the ENA records and builds `metadata.tsv`.
4. Counts the reads in each file and checks the count against ENA's.
5. Runs FastQC.
6. Reads the quality of each position along the reads and suggests where to cut.
7. You choose the cut values. The notebook trims and filters the reads and saves them in `clean/`.

**About unzipping.** The files stay as `.fastq.gz`. FastQC, fastplong and the tools in the next notebooks all read compressed files directly, so nothing needs unzipping, and that saves disk space.

Run the cells one at a time, from top to bottom, and read the output of each before moving on.

## 1. Imports

- `gzip` reads compressed `.gz` files.
- `json` reads JSON files, which is the format fastplong writes its report in.
- `csv` writes the metadata table as a tab-separated file.
- `zipfile` opens the `.zip` that FastQC writes.
- `subprocess` runs another program (FastQC, fastplong) from Python.
- `os`, `sys` and `shutil` are used once, to check the tools can be found.
- `xml.etree.ElementTree` reads XML, which is the format ENA's records come in.
- `date` gives today's date.
- `Path` builds file and folder paths.
- `requests` fetches web addresses.

Everything except `requests` comes with Python.

In [ ]:
import csv
import gzip
import json
import os
import shutil
import subprocess
import sys
import zipfile
import xml.etree.ElementTree as ET
from datetime import date
from pathlib import Path

import requests

## 2. Your input and the folders

`ACCESSION` is the same one you used in step 1. It tells the notebook which `data/` folder to use.

The notebook then makes three new folders inside it:

- `metadata/` for the ENA records.
- `qc/` for the quality reports.
- `clean/` for the trimmed and filtered reads.

`exist_ok=True` means: do not complain if the folder is already there.

In [ ]:
ACCESSION = "PRJEB40277"

DATA_FOLDER = Path("data") / ACCESSION
METADATA_FOLDER = DATA_FOLDER / "metadata"
QC_FOLDER = DATA_FOLDER / "qc"
CLEAN_FOLDER = DATA_FOLDER / "clean"

for folder in [METADATA_FOLDER, QC_FOLDER, CLEAN_FOLDER]:
    folder.mkdir(parents=True, exist_ok=True)

print("Working in:", DATA_FOLDER)

## 3. Check the tools can be found

FastQC and fastplong were installed into the conda environment by `environment.yml`. This cell checks that this notebook can find them.

The first line adds the folder of the Python you are running to the list of places the computer searches for programs (called the PATH). It makes the check work even if the notebook was started without activating the environment.

`shutil.which` gives the location of a program, or `None` if it cannot be found.

**Look at the output.** Both lines should show a location that contains `ls-use-case`. If you see `None`, the notebook is not using the `ls-use-case` kernel. Choose that kernel (top right of the notebook) and run the cell again.

In [ ]:
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

for tool in ["fastqc", "fastplong"]:
    print(tool, "->", shutil.which(tool))

## 4. List the FASTQ files

This cell finds every `.fastq.gz` file in the data folder.

The **run accession** (for example `ERR7112279`) is the first part of the file name, before the first `.`. This notebook uses it to name everything it makes, so any file can be traced back to its run.

This works because these files are single-end, one file per run. A paired-end run would have two files named `_1` and `_2`, and this notebook would need changing.

`sorted(...)` puts the files in alphabetical order. `.glob("*.fastq.gz")` means: every file whose name ends in `.fastq.gz`.

In [ ]:
fastq_files = sorted(DATA_FOLDER.glob("*.fastq.gz"))

for fastq_file in fastq_files:
    run = fastq_file.name.split(".")[0]
    print(run, fastq_file.name, round(fastq_file.stat().st_size / 1000000, 1), "MB")

print()
print("Number of files:", len(fastq_files))

## 5. Write a README into the data folder

Files with no explanation are a problem. This cell writes `README.txt` into the data folder. It says what every folder is for, and where the information about each file lives.

The folders for steps 3 to 5 do not exist yet. The notebooks for those steps create them.

In [ ]:
readme = """DATA FOLDER README

This folder holds the files for one ENA record. It was made by the notebooks in
https://github.com/ELIXIR-IE/eosc-gravity-genvidata

Folders:
  (this folder)  The raw FASTQ files downloaded from ENA (step 1). Never edited.
  metadata/      ENA's own records for each file (XML) and metadata.tsv, a table
                 with one row per FASTQ file (step 2).
  qc/            Quality reports from FastQC and fastplong (step 2).
  clean/         Reads after trimming and filtering (step 2).
  mapping/       Reads aligned to a reference, and the consensus sequence (step 3).
  assembly/      Output of the de novo assembly attempt (step 4).
  deposition/    Lineage calls, metadata and files prepared for submission (step 5).

Every file made by the notebooks starts with the run accession (for example
ERR7112279). Look that accession up in metadata/metadata.tsv to find the sample,
the study, the date and place of collection, and the sequencing method.
"""

(DATA_FOLDER / "README.txt").write_text(readme)

print("Written:", DATA_FOLDER / "README.txt")

## 6. Fetch ENA's records for each file

ENA keeps four linked records for each run:

- **run**: the sequencing run and its files.
- **experiment**: how the library was made and which instrument was used.
- **sample**: what was sequenced. Where and when it was collected, the host, the lab.
- **study**: the project the run belongs to.

The run record tells us the accessions of the other three. So the cell fetches the run first, reads the other accessions out of it, then fetches those.

Every record is saved exactly as ENA sent it, in `metadata/`, for example `ERR7112279.sample.xml`.

This uses the **ENA Browser API**, which is a different part of ENA from the Portal API used in step 1.

Lines worth explaining:

- `ET.fromstring(...)` turns the XML text into something Python can search.
- `.find("RUN/EXPERIMENT_REF")` looks for a `RUN` element and, inside it, an `EXPERIMENT_REF` element. `.get("accession")` reads the `accession` written on it.
- `.iter("XREF_LINK")` goes through every link in the record. Each link names a database (`DB`) and an ID.
- `f"{run}.{kind}.xml"` builds a file name by putting the values in the curly brackets into the text.

**Look at the output.** One line per run with its experiment, sample and study accessions.

In [ ]:
BROWSER_API = "https://www.ebi.ac.uk/ena/browser/api/xml/"

accessions = {}

for fastq_file in fastq_files:
    run = fastq_file.name.split(".")[0]

    response = requests.get(BROWSER_API + run, timeout=60)
    response.raise_for_status()
    (METADATA_FOLDER / (run + ".run.xml")).write_text(response.text)

    run_root = ET.fromstring(response.text)
    experiment = run_root.find("RUN/EXPERIMENT_REF").get("accession")

    for link in run_root.iter("XREF_LINK"):
        database = link.find("DB").text
        if database == "ENA-SAMPLE":
            sample = link.find("ID").text
        if database == "ENA-STUDY":
            study = link.find("ID").text

    for kind, accession in [("experiment", experiment), ("sample", sample), ("study", study)]:
        response = requests.get(BROWSER_API + accession, timeout=60)
        response.raise_for_status()
        (METADATA_FOLDER / f"{run}.{kind}.xml").write_text(response.text)

    accessions[run] = {"experiment": experiment, "sample": sample, "study": study}
    print(run, experiment, sample, study)

## 7. Build the metadata table

This cell reads the saved XML files and builds `metadata/metadata.tsv`. It has one row per FASTQ file. It is the one place to look up what a file is.

Each row holds:

- the run, experiment, sample and study accessions
- the file name
- who submitted the run, how the library was made, and which instrument was used
- the number of reads and bases ENA says the run has
- **every attribute of the sample**, each in a column that starts with `sample: `. This is where the collection date, the place, the host and the lab are.
- where the information came from, and the date it was fetched

The file is tab-separated (TSV). You can open it in a spreadsheet.

Lines worth explaining:

- `.iter("SAMPLE_ATTRIBUTE")` goes through every attribute of the sample. Each has a `TAG` (the name) and a `VALUE`.
- `.findtext("TAG")` gives the text of the `TAG` element, or nothing if there is none.
- `columns` collects every column name that appears in any row. Different samples can have different attributes, so a row may not have a value for every column. `restval=""` fills those gaps with an empty value.

**Look at the output.** The cell prints the first row, one column per line, so you can read what ENA says about the first file.

In [ ]:
today = date.today().isoformat()

rows = []

for fastq_file in fastq_files:
    run = fastq_file.name.split(".")[0]

    run_root = ET.parse(METADATA_FOLDER / f"{run}.run.xml").getroot()
    experiment_root = ET.parse(METADATA_FOLDER / f"{run}.experiment.xml").getroot()
    sample_root = ET.parse(METADATA_FOLDER / f"{run}.sample.xml").getroot()
    study_root = ET.parse(METADATA_FOLDER / f"{run}.study.xml").getroot()

    row = {}
    row["run_accession"] = run
    row["fastq_file"] = fastq_file.name
    row["experiment_accession"] = accessions[run]["experiment"]
    row["sample_accession"] = accessions[run]["sample"]
    row["study_accession"] = accessions[run]["study"]
    row["study_title"] = study_root.findtext(".//TITLE")
    row["submitted_by"] = run_root.find("RUN").get("center_name")
    row["instrument_model"] = experiment_root.findtext(".//INSTRUMENT_MODEL")
    row["library_strategy"] = experiment_root.findtext(".//LIBRARY_STRATEGY")
    row["library_protocol"] = experiment_root.findtext(".//LIBRARY_CONSTRUCTION_PROTOCOL")
    row["library_description"] = experiment_root.findtext(".//DESIGN_DESCRIPTION")
    row["sample_title"] = sample_root.findtext(".//TITLE")

    for attribute in run_root.iter("RUN_ATTRIBUTE"):
        if attribute.findtext("TAG") == "ENA-SPOT-COUNT":
            row["ena_read_count"] = attribute.findtext("VALUE")
        if attribute.findtext("TAG") == "ENA-BASE-COUNT":
            row["ena_base_count"] = attribute.findtext("VALUE")

    for attribute in sample_root.iter("SAMPLE_ATTRIBUTE"):
        row["sample: " + attribute.findtext("TAG")] = attribute.findtext("VALUE")

    row["source"] = BROWSER_API + run
    row["fetched_on"] = today

    rows.append(row)

columns = []
for row in rows:
    for column in row:
        if column not in columns:
            columns.append(column)

with open(METADATA_FOLDER / "metadata.tsv", "w", newline="") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=columns, delimiter="\t", restval="")
    writer.writeheader()
    writer.writerows(rows)

print("Written:", METADATA_FOLDER / "metadata.tsv")
print("Rows:", len(rows), " Columns:", len(columns))
print()

for column in columns:
    print(column, "=", rows[0].get(column, ""))

## 8. Count the reads and check them against ENA

ENA's record says how many reads and bases each run has. This cell counts them in the file you downloaded. If the numbers match, the file and the record belong together.

Lines worth explaining:

- `enumerate(input_file)` gives each line together with its number, starting from 0.
- `line_number % 4` is the remainder after dividing by 4. In a FASTQ file every read takes four lines, so `% 4 == 1` picks the second line of each read, which is the sequence.
- `len(line.strip())` is the length of the sequence.

**Look at the output.** Every line should end in `OK`.

In [ ]:
for row in rows:

    fastq_file = DATA_FOLDER / row["fastq_file"]

    read_count = 0
    base_count = 0

    with gzip.open(fastq_file, "rt") as input_file:
        for line_number, line in enumerate(input_file):
            if line_number % 4 == 1:
                read_count = read_count + 1
                base_count = base_count + len(line.strip())

    if read_count == int(row["ena_read_count"]) and base_count == int(row["ena_base_count"]):
        verdict = "OK"
    else:
        verdict = "DIFFERENT"

    print(row["run_accession"], "reads:", read_count, "bases:", base_count, "mean length:", round(base_count / read_count), verdict)

## 9. Run FastQC

FastQC is the standard first look at sequencing reads. For each file it writes an HTML report with plots, and a `.zip` with the same results as text.

`subprocess.run(command)` runs a program as if you had typed it in a terminal. The command is a list: the program name first, then each option as its own item.

- `--outdir` is the folder to write the reports to. It must already exist (it does, from step 2).
- `-t 4` means use 4 processor threads.

**Things to know about FastQC on these reads.**

- FastQC was designed for short Illumina reads. These are Nanopore reads, about 1,000 bases long. The plots still work, but the PASS / WARN / FAIL labels are tuned for Illumina and will often warn about things that are normal for Nanopore.
- FastQC groups the later positions of a long read into bins, so the quality plot is coarse.

The cell runs for about a minute. FastQC prints its progress below the cell.

**Look at the output.** `Exit code: 0` means it finished without errors.

In [ ]:
command = ["fastqc", "--outdir", str(QC_FOLDER), "-t", "4"]

for fastq_file in fastq_files:
    command.append(str(fastq_file))

result = subprocess.run(command)

print("Exit code:", result.returncode)

## 10. FastQC verdicts

Each FastQC report is made of modules, and each module gets PASS, WARN or FAIL. This cell lists, for every run, the modules that did **not** pass.

Remember the warning above: for Nanopore reads some of these verdicts are expected and are not a sign of a problem.

To see the plots, open the HTML reports in a browser. The cell prints where they are.

Lines worth explaining:

- `zipfile.ZipFile(...)` opens a `.zip` file. `.read(...)` reads one file out of it without unpacking it onto the disk.
- `summary.txt` inside the zip has one line per module: the verdict, a tab, the module name, a tab, the file name.

In [ ]:
for fastq_file in fastq_files:
    run = fastq_file.name.split(".")[0]

    with zipfile.ZipFile(QC_FOLDER / f"{run}_fastqc.zip") as zip_file:
        summary = zip_file.read(f"{run}_fastqc/summary.txt").decode()

    print(run)
    for line in summary.strip().split("\n"):
        verdict, module, file_name = line.split("\t")
        if verdict != "PASS":
            print("   ", verdict, module)

    print("    report:", QC_FOLDER / f"{run}_fastqc.html")

## 11. Measure the quality along the reads with fastplong

FastQC gives pictures. To work with the numbers in Python we use a second tool, **fastplong**. It is the long-read version of fastp.

Here it only measures. Three switches turn its filtering off, so nothing is removed:

- `-A` no adapter trimming
- `-Q` no quality filtering
- `-L` no length filtering

`-j` and `-h` are the names of the JSON and HTML reports it writes.

**Look at the output.** One line per run when it finishes.

In [ ]:
for fastq_file in fastq_files:
    run = fastq_file.name.split(".")[0]

    command = [
        "fastplong",
        "-i", str(fastq_file),
        "-j", str(QC_FOLDER / f"{run}.before.json"),
        "-h", str(QC_FOLDER / f"{run}.before.html"),
        "-A", "-Q", "-L",
    ]

    result = subprocess.run(command, capture_output=True, text=True)

    print(run, "exit code:", result.returncode)

## 12. Quality at the start of the reads

The quality of a base is a **Phred score**. Q10 means a 1 in 10 chance the base is wrong. Q20 is 1 in 100. Q30 is 1 in 1,000.

fastplong's report contains the average quality at every position counted from the start of the read (position 0 is the first base). This cell prints it at a handful of positions.

**Look at the output.** Quality is usually poor at the very first bases and then climbs. The position where it first becomes good is a sensible place to cut.

In [ ]:
POSITIONS = [0, 1, 2, 3, 5, 10, 15, 20, 30, 50, 100, 500, 1000]

for fastq_file in fastq_files:
    run = fastq_file.name.split(".")[0]

    with open(QC_FOLDER / f"{run}.before.json") as input_file:
        report = json.load(input_file)

    curve = report["read_before_filtering"]["quality_curves"]["mean"]

    print(run)
    for position in POSITIONS:
        if position < len(curve):
            print("    position", position, "average quality", round(curve[position], 1))

## 13. Where to cut: a suggestion

**No tool gives a recommended cut point for Nanopore reads.** This is a known gap. The usual practice is to look at the plots, pick values, and keep a length range and a minimum average quality. This cell does the looking for you and prints a suggestion. You then decide.

- `QUALITY_TARGET` is the average quality you want the first and last base to reach. Q20 is a common choice. You can change it.
- **Start of the read.** The cell uses the average-quality numbers from step 12. It finds the first position where the average reaches the target.
- **End of the read.** The numbers from step 12 cannot be used here, because reads have different lengths, so "position 900" is the end of some reads and the middle of others. The cell instead reads the first 5,000 reads itself, turns each quality character into a number, lines the reads up by their **last** base, and averages position by position, counting back from the end.
- For comparison the cell prints what **ENA's own pipeline** does: it cuts a fixed 30 bases from both ends of every read.

Lines worth explaining:

- Each quality character is a Phred score written as a letter. `ord(character) - 33` turns it into a number.
- `scores[::-1]` is the list backwards, so the last base comes first.
- This is a plain average of Phred scores. That is how the tools draw their plots, but it is not the same as the average error rate.

**Look at the output.** One line per run.

In [ ]:
QUALITY_TARGET = 20
READS_TO_CHECK = 5000
POSITIONS_FROM_END = 300

for fastq_file in fastq_files:
    run = fastq_file.name.split(".")[0]

    with open(QC_FOLDER / f"{run}.before.json") as input_file:
        report = json.load(input_file)

    curve = report["read_before_filtering"]["quality_curves"]["mean"]

    suggested_start = 0
    for position, quality in enumerate(curve):
        if quality >= QUALITY_TARGET:
            suggested_start = position
            break

    totals = [0] * POSITIONS_FROM_END
    counts = [0] * POSITIONS_FROM_END
    reads_seen = 0

    with gzip.open(fastq_file, "rt") as input_file:
        for line_number, line in enumerate(input_file):
            if line_number % 4 == 3:
                scores = [ord(character) - 33 for character in line.strip()]
                backwards = scores[::-1]
                for index, score in enumerate(backwards[:POSITIONS_FROM_END]):
                    totals[index] = totals[index] + score
                    counts[index] = counts[index] + 1
                reads_seen = reads_seen + 1
                if reads_seen == READS_TO_CHECK:
                    break

    suggested_end = 0
    for index in range(POSITIONS_FROM_END):
        if totals[index] / counts[index] >= QUALITY_TARGET:
            suggested_end = index
            break

    print(run, "suggested cut at start:", suggested_start, " at end:", suggested_end, " (ENA pipeline: 30 and 30)")

## 14. Your input: trim and filter

Choose the values below. They start at settings that follow ENA's own pipeline. Change them if the suggestions above or the FastQC plots tell you to.

- `TRIM_START` and `TRIM_END`: how many bases to cut from the start and end of every read.
- `MIN_LENGTH` and `MAX_LENGTH`: reads outside this length range are removed. This data is amplicon sequencing, where every read should be about as long as one amplicon. Reads far shorter or longer are usually junk. The reads here run from 250 to 1,494 bases.
- `MIN_MEAN_QUALITY`: reads whose average quality is below this are removed. ARTIC, the standard SARS-CoV-2 amplicon protocol, uses 7 as its default.

The cell then runs fastplong again, this time keeping the cleaned reads. It writes:

- `clean/<run>.clean.fastq.gz` the cleaned reads
- `qc/<run>.after.json` and `.html` a report on the cleaned reads

`-f` and `-t` are fastplong's names for trim-front and trim-tail. `-l` is the minimum length, `--length_limit` the maximum, and `-m` the minimum average quality. `-A` still turns off adapter trimming, because adapters have already been removed from these files.

**Look at the output.** For each run: how many reads went in, how many came out, and why the others were removed.

In [ ]:
TRIM_START = 30
TRIM_END = 30
MIN_LENGTH = 200
MAX_LENGTH = 1500
MIN_MEAN_QUALITY = 10

for fastq_file in fastq_files:
    run = fastq_file.name.split(".")[0]

    command = [
        "fastplong",
        "-i", str(fastq_file),
        "-o", str(CLEAN_FOLDER / f"{run}.clean.fastq.gz"),
        "-j", str(QC_FOLDER / f"{run}.after.json"),
        "-h", str(QC_FOLDER / f"{run}.after.html"),
        "-A",
        "-f", str(TRIM_START),
        "-t", str(TRIM_END),
        "-l", str(MIN_LENGTH),
        "--length_limit", str(MAX_LENGTH),
        "-m", str(MIN_MEAN_QUALITY),
    ]

    result = subprocess.run(command, capture_output=True, text=True)

    with open(QC_FOLDER / f"{run}.after.json") as input_file:
        report = json.load(input_file)

    reads_in = report["read_before_filtering"]["total_reads"]
    reads_out = report["read_after_filtering"]["total_reads"]
    removed = report["filtering_result"]

    print(run, "reads in:", reads_in, "reads out:", reads_out)
    print("    removed for low quality:", removed["low_quality_reads"],
          " too short:", removed["too_short_reads"],
          " too long:", removed["too_long_reads"])

## 15. Were human reads removed?

You expected the host (human) reads to have been removed already. This is what can and cannot be told.

- **ENA's rule.** ENA asks for human reads to be removed before SARS-CoV-2 reads are submitted. The Swedish Pathogens Portal says the same.
- **ENA does not record it.** There is no field in ENA's run, experiment or sample records that says host reads were removed. The sample checklist for viruses (`ERC000033`) has none either. The submitter's word is all there is.
- **What we can check.** If human reads were left in, a share of the reads would not match the virus. Step 3 maps the reads to the virus and reports the percentage that match. Close to 100% means very little else is in the files.
- **What that does not prove.** Reads that do not map are not necessarily human. A tool that looks for human reads directly, such as the NCBI human read scrubber, needs a database of about 1 GB, so it is left out of this use case.

Nothing to run in this cell. Step 3 is next.

**Checkpoint.** When this notebook has run you should have:

- `metadata/metadata.tsv` and the XML records
- `qc/` with FastQC and fastplong reports
- `clean/` with one cleaned file per run